In [1]:
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl

from tqdm.auto import tqdm

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

from multiprocess import Pool

from scipy import stats


In [3]:
workspace_root = os.getcwd()
sys.path.insert(0, workspace_root + "/../../../")
import pyanalib.pandas_helpers as ph
from makedf.util import *

sys.path.insert(0, workspace_root + "/../")
import kinematics
import gump_cuts as gc
import loaddf
import syst
import importlib

In [4]:
PLOTDIR = "/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/MCDataComp"

DOSAVE = False

import os
os.makedirs(PLOTDIR, exist_ok=True)
os.makedirs(PLOTDIR + "/png", exist_ok=True)
os.makedirs(PLOTDIR + "/pdf", exist_ok=True)

In [5]:
DETECTOR = "ICARUS Run4"
INCLUDE_DIRT = True

In [6]:
# Just a big list of files, needs to be updated
DF_DIR = "/exp/sbnd/data/users/nrowe/GUMP/sbn-rewgted-6/"

if DETECTOR == "ICARUS Run2":
    ONBEAM = DF_DIR + "ICARUSRun2_SpringRun2BNB_unblind_prescaled.df"
    OFFBEAM = [DF_DIR + f"ICARUSRun2_BNBoff_uncalo_unblind_{i}.df" for i in range(60)]
    
    ONBEAMPOT = DF_DIR + "ICARUSRun2_SpringRun2BNB_unblind_prescaled_POT.df"

    MC_FILES = [DF_DIR + "ICARUSRun2_SpringMCOverlay_rewgt.df"]
    
    DIRT_FILE = DF_DIR + "ICARUSRun2_Spring_Overlay_Dirt.df"
    
    DETVAR_FILES = []
    DETVAR_NAMES = []
elif DETECTOR == "ICARUS Run4":
    ONBEAM = DF_DIR + "ICARUSRun4_SpringRun4BNB_unblind_prescaled.df"
    OFFBEAM = [DF_DIR + f"ICARUSRun4_BNBoff_{i}.df" for i in range(30)]
    
    ONBEAMPOT = DF_DIR + "ICARUSRun4_SpringRun4BNB_unblind_prescaled_POT.df"

    MC_FILES = [DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_%i.df" % i for i in range(3)]
    DIRT_FILE = None
    DIRT_FILE = DF_DIR + "ICARUSRun4_Spring_Overlay_Dirt.df"
    
    DETVAR_FILES = []
    DETVAR_NAMES = []
elif DETECTOR == "SBND": 
    ONBEAM = DF_DIR + "SBND_SpringBNBData_Dev.df"
    OFFBEAM = DF_DIR + "SBND_SpringBNBOffData_5000.df"

    MC_FILES = [DF_DIR + "SBND_SpringMC_rewgt_5_%i.df" % i for i in range(10)]
    DIRT_FILE = DF_DIR + "SBND_SpringLowEMC.df"

    DETVAR_FILES = [
        DF_DIR + "SBND_SpringMC_Nom.df",
        DF_DIR + "SBND_SpringMC_WMXThetaXW.df",
        DF_DIR + "SBND_SpringMC_WMYZ.df",
        DF_DIR + "SBND_SpringMC_2xSCE.df",
    ]
    DETVAR_NAMES = ["Nominal",
                    "WM $X\\theta_{xw}$", "WM $YZ$", 
                    "2x SCE"]

In [7]:
import h5py

def read_dfs(file, key):
    with h5py.File(file, "r") as f:
        keys = [k for k in f.keys() if k.startswith(key)]
        return pd.concat([pd.read_hdf(file, k) for k in keys])

In [8]:
# get gate based scale factor for offbeam vs onbeam

if "ICARUS" in DETECTOR:
    ngates_ON = read_dfs(ONBEAM, "trig").gate_delta.sum()*(1-1/100.)
    ngates_OFF = sum([read_dfs(O, "trig").gate_delta.sum()*(1-1/20.) for O in OFFBEAM])
    
    OFF_w = ngates_ON / ngates_OFF
elif "SBND" in DETECTOR:
    ngates_ON = read_dfs(ONBEAM, "bnb").shape[0]
    ngates_OFF = read_dfs(OFFBEAM, "hdr").noffbeambnb.sum()

    f_factor = 0.0754
    OFF_w = (1. - f_factor) * (ngates_ON) / (ngates_OFF)

In [9]:
if "ICARUS" in DETECTOR:
    # POT = pd.read_hdf(ONBEAMPOT).pot.sum()*1e12
    POT = read_dfs(ONBEAM, "hdr").merge(pd.read_hdf(ONBEAMPOT), left_index=True, right_index=True, how="left").pot_y.sum()*1e12
elif "SBND" in DETECTOR:
    POT = read_dfs(ONBEAM, "bnb").TOR875.sum()

In [10]:
def FV(df):    
    det = DETECTOR.split(" ")[0]
    ret = gc.slcfv_cut(df, det) 
    return ret

def flash_cut(df):    
    det = DETECTOR.split(" ")[0]
    ret = gc.flash_cut(df, det)
    return ret

def containment_cut(df):    
    det = DETECTOR.split(" ")[0]
    ret = gc.mufv_cut(df, det) & gc.pfv_cut(df, det) 
    return ret

def pid_cut(df):    
    ret = gc.pid_cut(df.mu_chi2_of_mu_cand, df.mu_chi2_of_prot_cand,
                  df.prot_chi2_of_mu_cand, df.prot_chi2_of_prot_cand,
                  df.mu_len)
    return ret

In [11]:
# LOAD MC CV SAMPLES
if DETECTOR == "ICARUS Run4":
    ncores = 1
else:
    ncores = 10
df, match, mcpot = loaddf.loadl(MC_FILES, njob=min(len(MC_FILES), ncores), reweight_aFF=True, preselection=FV, detector=DETECTOR)

  0%|          | 0/3 [00:00<?, ?it/s]

In [12]:
# LOAD DETECTOR VARIATION SAMPLES

if len(DETVAR_FILES) > 0:
    detvars, detvars_match, detvar_pots = zip(*[loaddf.load(f, include_syst=False, detector=DETECTOR) for f in tqdm(DETVAR_FILES)])
    detvars, detvar_pots = loaddf.match_common_evts(detvars_match, detvars, detvar_pots)
else:
    detvars = []
    detvar_pots = []

In [13]:
loaddf.scale_pot(df, mcpot, POT)
for i in range(len(detvars)):
    print(DETVAR_NAMES[i])
    loaddf.scale_pot(detvars[i], detvar_pots[i], POT)

(np.float32(6.307889e+20), np.float64(0.014179340987850203))

In [14]:
# This removes things in the AV, so that our dirt sample is actually a dirt sample
def ICARUS_dirtcut(df):
    # mbox from CV sample
    xlo = -378.49
    ylo = -191.86
    zlo = -904.950652270838

    xhi = 378.49
    yhi = 144.96
    zhi = 904.950652270838
    vtx = pd.DataFrame({'x': df.true_vtx_x,
                       'y': df.true_vtx_y,
                       'z': df.true_vtx_z}, index=df.index)

    return ~((vtx.x > xlo) & (vtx.x < xhi) & (vtx.y > ylo) & (vtx.y < yhi) & (vtx.z > zlo) & (vtx.z < zhi))

if INCLUDE_DIRT and DIRT_FILE is not None:
    dirt, dirtmatch, dirtpot = loaddf.load(DIRT_FILE, include_syst=False, preselection=FV, detector=DETECTOR)

    loaddf.scale_pot(dirt, dirtpot, POT)
    
    if DETECTOR == "ICARUS":
        dirt = dirt[ICARUS_dirtcut(dirt)]

    dirt["crthit"] = False
    dirt["dirt"] = True
    df["dirt"] = False
elif INCLUDE_DIRT:
    df["dirt"] = False

(np.float32(9.350443e+19), np.float64(0.09565504498093605))

In [15]:
if INCLUDE_DIRT and DIRT_FILE is not None:
    # reset df if needed
    df = df[~df.dirt]
    
    # add dirt to the CV, and also include it (below) as a 100 syst unc.
    df = pd.concat([df, dirt])

    # keep copy of non-dirt around for chi2 systematic variation CV
    df0 = df.loc[~df.dirt, [c for c in df.columns if "univ" not in c]]
    
    # disable systematic weights associated with dirt events
    for c in df.columns:
        if "univ" in c:
            df[c] = df[c].fillna(1)
            
    # also add it to the detector variations, for consistency
    for i in range(len(detvars)):
        detvars[i] = pd.concat([detvars[i], dirt])
elif INCLUDE_DIRT:
    # keep copy of non-dirt around for chi2 systematic variation CV
    df0 = df.loc[~df.dirt, [c for c in df.columns if "univ" not in c]]

In [16]:
# ADD IN PID VARIATIONS
def v_variation(df, setvars):
    df = df[[c for c in df.columns if "univ" not in c]].copy()
    for (new, old) in setvars:
        df[new] = df[old]
    return df

def v_chi2smear(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2smear13_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2smear13_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2smear13_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2smear13_of_prot_cand"),
    ]
    return v_variation(df, setvars)


def v_chi2hi(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2hi_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2hi_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2hi_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2hi_of_prot_cand"),
    ]
    return v_variation(df, setvars)

def v_chi2lo(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2lo_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2lo_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2lo_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2lo_of_prot_cand"),
    ]
    return v_variation(df, setvars)

def v_chi22hi(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi22hi_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi22hi_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi22hi_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi22hi_of_prot_cand"),
    ]
    return v_variation(df, setvars)

def v_chi22lo(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi22lo_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi22lo_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi22lo_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi22lo_of_prot_cand"),
    ]
    return v_variation(df, setvars)

In [17]:
chi2_detvars = [[v_chi2hi(df0), v_chi2lo(df0)], v_chi2smear(df0), ]
chi2_2x_detvars = [[v_chi22hi(df0), v_chi22lo(df0)], v_chi2smear(df0), ]

In [18]:
def TrueAV(df, det):
    if det == "ICARUS Run4":
        df['Run'] = 4
    elif det == "ICARUS Run2":
        df['Run'] = 2    
    elif det == "SBND":
        df['Run'] = 1
    else:
        print('unclear run!!!')
        
    vtx = pd.DataFrame({'Run': df.Run,
                       'x': df.true_vtx_x,
                       'y': df.true_vtx_y,
                       'z': df.true_vtx_z}, index=df.index)
    return gc._fv_cut(vtx, det, 0, 0, 0, 0)

def OOAV(df):
    det = DETECTOR.split(" ")[0]
    return ~np.isnan(df.true_vtx_x) & ~TrueAV(df, det)


df_ooav = df[OOAV(df)].copy()

unclear run!!!


In [19]:
ONdf,_,_ = loaddf.load(ONBEAM, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV, match_Enu=False)
if "ICARUS" in DETECTOR:
    OFFdf,_,_ = loaddf.loadl(OFFBEAM, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV, match_Enu=False)
else:
    OFFdf,_,_ = loaddf.load(OFFBEAM, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV, match_Enu=False)

  0%|          | 0/30 [00:00<?, ?it/s]

In [20]:
mode_list = [1, 10, 0]

mode_labels = ["Cosmic", "Dirt", 'Other $\\nu$', '$\\nu_\\mu$ CC RES', '$\\nu_\\mu$ CC MEC', '$\\nu_\\mu$ CC QE']
mode_colors = ["#95af8b", "#43140b", "#c89648", "#1e3f54", "#d54c28", "#315031"]

def nuFV(df):
    det = DETECTOR.split(" ")[0]

    vtx = pd.DataFrame({
        "Run": df.Run,
        "x": df.true_vtx_x,
        "y": df.true_vtx_y,
        "z": df.true_vtx_z,
    })
    return ~np.isnan(df.true_vtx_x) & gc._fv_cut(vtx, det, 0, 0, 0, 0)

def breakdown_mode(var, df):
    """Break down variable by interaction mode."""
    numu_cc = (np.abs(df.true_nu_pdg) == 14) & (df.true_isnc == False)
    fid = nuFV(df)
    fid = ~df.dirt

    ret = [
        var[np.isnan(df.genie_mode)],
        var[~fid & ~np.isnan(df.genie_mode)],
        var[(~np.any([df.genie_mode == i for i in mode_list], axis=0) | ~numu_cc) & fid & ~np.isnan(df.genie_mode)]
    ] +\
        [var[(df.genie_mode == i) & numu_cc & fid] for i in mode_list]
        
    return ret

In [21]:
FONTSIZE = 14
HAWKS_COLORS = ["#315031", "#d54c28", "#1e3f54", "#c89648", "#43140b", "#95af8b"]

def add_style(ax, xlabel, title="", det="ICARUS"):
    ax.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.set_xlabel(xlabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_ylabel('Area Normalized', fontsize=FONTSIZE, fontweight='bold')
    ax.set_title(f"$\\bf{{{det}}}$  {title}", fontsize=FONTSIZE+2)
    ax.legend(fontsize=FONTSIZE)


In [22]:
def f_chi2(NMC, Ndata, cov):
    # ignore singular entries
    which_bin = NMC > 0

    NMC = NMC[which_bin]
    Ndata = Ndata[which_bin]
    cov = cov[which_bin, :]
    cov = cov[:, which_bin]

    delta = NMC - Ndata
    try:
        cov_inv = np.linalg.inv(cov)
    except np.linalg.LinAlgError as _:
        return -1, which_bin.sum()
        
    return delta@cov_inv@delta, which_bin.sum()

In [23]:
# def simple_cosmic_rej(df):
#     is_spine = "SPINE" in DETECTOR
#     return FV(df) & (df.nu_score > 0.6)

# def crtveto(df):
#     return FV(df) & ~df.crthit

# def twoprong_cut(df):
#     return FV(df) & np.isnan(df.other_shw_length) & np.isnan(df.other_trk_length)

# def pid_cut(df):
#     is_spine = "SPINE" in DETECTOR
#     if not is_spine:
#         return twoprong_cut(df) & gc.pid_cut_df(df)
#     else:
#         return twoprong_cut(df) & (df.prot_chi2_of_prot_cand > 0.6) & (df.mu_chi2_of_mu_cand > 0.6)


In [24]:
if DETECTOR == "SBND":
    x_range = np.linspace(-200, 200, 21)
    y_range = np.linspace(-200, 200, 21)
    z_range = np.linspace(0, 500, 21)
else:
    x_range = np.linspace(-360, 360, 21)
    y_range = np.linspace(-182, 135, 21)
    z_range = np.linspace(-895, 895, 21)

In [25]:
extravarnames = [
    "mu_costh",
    "mu_phi",
    "p_costh",
    "p_phi",
]

extravars = [
    lambda d: np.cos(d.mu_dir_z),
    lambda d: np.arctan2(d.mu_dir_x, d.mu_dir_y)*180/np.pi,
    lambda d: np.cos(d.p_dir_z),
    lambda d: np.arctan2(d.p_dir_x, d.p_dir_y)*180/np.pi,
]

In [26]:
for v, vname in zip(extravars, extravarnames):
    df[vname] = v(df)
    df0[vname] = v(df0)
    df_ooav[vname] = v(df_ooav)
    ONdf[vname] = v(ONdf)
    OFFdf[vname] = v(OFFdf)

    if INCLUDE_DIRT and DIRT_FILE is not None:
        dirt[vname] = v(dirt)

    for i in range(len(detvars)):
        detvars[i][vname] = v(detvars[i])

    for i in range(len(chi2_detvars)):
        if isinstance(chi2_detvars[i], list):
            for j in range(len(chi2_detvars[i])):
                chi2_detvars[i][j][vname] = v(chi2_detvars[i][j])
        else:
            chi2_detvars[i][vname] = v(chi2_detvars[i])

In [27]:
df = gc.add_opening_angle_mu_p(df)
df0 = gc.add_opening_angle_mu_p(df0)
df_ooav = gc.add_opening_angle_mu_p(df_ooav)
ONdf = gc.add_opening_angle_mu_p(ONdf)
OFFdf = gc.add_opening_angle_mu_p(OFFdf)

In [28]:
def step_0(df):
    return (df.nu_E_calo > -999.)

def step_1(df):
    return step_0(df) & FV(df)

def step_2(df):
    return step_1(df) & flash_cut(df)
    
def step_3(df):
    return step_2(df) & gc.cosmic_cut(df)

def step_4(df):
    return step_3(df) & gc.twoprong_cut(df)
    
def step_5(df):
    return step_4(df) & containment_cut(df)

def step_6(df):
    return step_4(df) & pid_cut(df)

In [29]:
cuts = [
    step_0,
    step_1,
    step_2,
    step_3,
    step_4,
    step_5,
    step_6
]

cutnames = [
    "No Cuts",
    "FV Cut Step",
    "Flash Cut Step",
    "Cosmic Cut Step",
    "Two Prong Cut Step",
    "Containment Cut Step",
    "PID Cut Step"
]

plotvars = [
    ["slc_vtx_x", "slc_vtx_y", "slc_vtx_z"],
    ["flash_maxpe"],
    ["mu_p_opening_angle_deg", "nu_score"],
    ["other_trk_length", "other_shw_length"],
    ["mu_end_x", "mu_end_y", "mu_end_z", "p_end_x", "p_end_y", "p_end_z"],
    ["mu_chi2_of_prot_cand", "prot_chi2_of_prot_cand", "mu_chi2_of_mu_cand", "prot_chi2_of_mu_cand", "mu_len"]
]

labels = [
    ["Vertex X [cm]", "Vertex Y [cm]", "Vertex Z [cm]"],
    ["Max In-Time Flash Total PE"],
    ["Muon-Proton Opening Angle [degrees]", "$\\nu$ Score"],
    ["Maximum Third Track Length [cm]", "Maximum Shower Length [cm]"],
    ["Muon End X [cm]", "Muon End Y [cm]", "Muon End Z [cm]", "Proton End X [cm]", "Proton End Y [cm]", "Proton End Z [cm]"],
    ["Proton Cand. $\\mu$-like PID", "Proton Cand. $p$-like PID", "Muon Cand. $\\mu$-like PID", "Muon Cand. $p$-like PID", "Muon Cand. Length [cm]"]
]

bins = [
    [x_range, y_range, z_range],
    [np.linspace(0, 40000, 20)],
    [np.linspace(0., 180., 11), np.linspace(0.25, 0.75, 11)],
    [np.array([-5] + list(np.linspace(0, 20, 5))), np.array([-10] + list(np.linspace(0, 100, 11)))],
    [x_range, y_range, z_range, x_range, y_range, z_range],
    [np.linspace(0, 60, 13), np.linspace(0, 120, 9), np.linspace(0, 60, 13), np.linspace(0, 300, 21), np.linspace(0, 100, 21)]
]

for c, cname in zip(cuts, cutnames):
    df[cname] = c(df)
    df0[cname] = c(df0)
    df_ooav[cname] = c(df_ooav)
    ONdf[cname] = c(ONdf)
    OFFdf[cname] = c(OFFdf)
    dirt[cname] = c(dirt)
    
    for i in range(len(detvars)):
        detvars[i][cname] = c(detvars[i])
        
    for i in range(len(chi2_detvars)):
        if isinstance(chi2_detvars[i], list):
            for j in range(len(chi2_detvars[i])):
                chi2_detvars[i][j][cname] = c(chi2_detvars[i][j])
        else:
            chi2_detvars[i][cname] = c(chi2_detvars[i])
            
    for i in range(len(chi2_2x_detvars)):
        if isinstance(chi2_2x_detvars[i], list):
            for j in range(len(chi2_2x_detvars[i])):
                chi2_2x_detvars[i][j][cname] = c(chi2_2x_detvars[i][j])
        else:
            chi2_2x_detvars[i][cname] = c(chi2_2x_detvars[i])

In [31]:
pot_scales = {'ps1': 0.5, 'ms1': -0.525805, 'ps2': 1.0, 'ms2': -1.12726, 'ps3': 1.5, 'ms3': -1.7286, 'cv': 0.0}

# Define systematic uncertainties
if "ICARUS" in DETECTOR:
    systematics = syst.SystematicList([
        loaddf.FluxSystematic(df),
        loaddf.XSecSystematic(df),
        loaddf.POTSystematic(df),
        syst.SystematicList(
            [syst.SampleSystematic(d, cvdf=df0) for d in chi2_detvars]),
        syst.SystSampleSystematic(df_ooav)])
else:
    systematics = syst.SystematicList([
        loaddf.FluxSystematic(df),
        loaddf.XSecSystematic(df),
        loaddf.POTSystematic(df),
        syst.SystematicList(
            [syst.SampleSystematic(d, cvdf=detvars[0]) for d in detvars[1:]] +
            [syst.SampleSystematic(d, cvdf=df0) for d in chi2_detvars]),
        syst.SystSampleSystematic(df_ooav)])

In [32]:
def make_plot_data(var, bins, cut, mc_weight, breakdown, areanorm, breakdown_labels, breakdown_colors, xlabel, title, 
                   det="ICARUS", fillna=np.nan, syst=systematics):
    
    pvars = breakdown(df.loc[df[cut], var].fillna(fillna), df[df[cut]])
    weights = breakdown(df.loc[df[cut], mc_weight], df[df[cut]])
    NMC_breakdown = []
    for pvar, w in zip(pvars, weights):    
        thisNMC, bins = np.histogram(pvar, bins=bins, weights=w)
        NMC_breakdown.append(thisNMC)
        
    NMC,_ = np.histogram(df.loc[df[cut], var].fillna(fillna), bins=bins, weights=df.loc[df[cut], mc_weight])
    NMC_abs = NMC
    if areanorm:
        diff = (bins[1:] - bins[:-1])
        norm = np.sum(NMC*diff)
        if norm > 1e-5:
            NMC = NMC / norm
            for i in range(len(NMC_breakdown)):
                NMC_breakdown[i] = NMC_breakdown[i] / norm

    NMC_breakdown = np.array(NMC_breakdown)
    NON,_ = np.histogram(ONdf.loc[ONdf[cut], var].fillna(fillna), bins=bins)
    NOff,_ = np.histogram(OFFdf.loc[OFFdf[cut], var].fillna(fillna), bins=bins)

    N = NON - NOff*OFF_w
    Nerr = np.sqrt(NON + NOff*OFF_w**2)
    if areanorm:
        diff = (bins[1:] - bins[:-1])
        
        norm = np.sum(N*diff)
        if norm > 1e-5:
            N = N / norm
            Nerr = Nerr / norm
    
    cov = syst.cov(var, cut, bins, NMC_abs, shapeonly=areanorm, fillna=fillna)
    err = np.sqrt(np.diag(cov))

    cov_w_stat = cov + np.diag(Nerr**2) # add stat uncertainty
    chi2, ndof = f_chi2(NMC, N, cov_w_stat)

    return {
        "det": det,
        "title": title,
        "xlabel": xlabel,
        "bins": bins,
        "areanorm": areanorm,
        "breakdown_labels": breakdown_labels,
        "breakdown_colors": breakdown_colors,
        "NMC_breakdown": NMC_breakdown,
        "NMC_total": NMC,
        "NData": N,
        "NDataErr": Nerr,
        "cov": cov,
        "cov_w_stat": cov_w_stat,
        "chi2": chi2,
        "ndof": ndof,
        "POT": POT
    }


In [33]:
def ratio_plot(plt, plotdata):
    fig, (ax0, ax1) = plt.subplots(2, 1, height_ratios=[3, 1], sharex=True)
    bins = plotdata["bins"]
    centers = (bins[:-1] + bins[1:])/2

    NMC_breakdown = plotdata["NMC_breakdown"]
    fill = np.array([centers for _ in range(NMC_breakdown.shape[0])]).T
    ax0.hist(fill, bins=bins, stacked=True, label=plotdata["breakdown_labels"],
                    color=plotdata["breakdown_colors"], weights=NMC_breakdown.T)

    NData = plotdata["NData"]
    NDataErr = plotdata["NDataErr"]
    line = ax0.errorbar(centers, NData, NDataErr, color="black", linestyle="none", marker=".")

    NMC = plotdata["NMC_total"]
    err = np.sqrt(np.diag(plotdata["cov"]))
    ax0.fill_between(bins[:-1], NMC+err, NMC-err, facecolor="none", hatch="//", edgecolor="gray", linewidth=0.0, step="post")

    if "Maximum" in plotdata["xlabel"]:
        ax0.set_yscale("log")

    ax1.errorbar(centers, NData/NMC, NDataErr/NMC, color="black", linestyle="none", marker=".")
    ax1.set_ylim([0.5, 1.5])
    ax1.axhline([1], color="red", linestyle="--")
    ax1.fill_between(bins[:-1], 1+err/NMC, 1-err/NMC, facecolor="none", hatch="//", edgecolor="gray", linewidth=0.0, step="post")

    ax0.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    ax1.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    for spine in ax0.spines.values():
        spine.set_linewidth(1.5)
    ax1.set_xlabel(plotdata["xlabel"], fontsize=FONTSIZE, fontweight='bold')
    
    if plotdata["areanorm"]:
        ax0.set_ylabel('Area Normalized' % (plotdata["POT"]/1e19), fontsize=FONTSIZE, fontweight='bold')
    else:
        ax0.set_ylabel('Events / %.1f$\\times 10^{19}$ POT' % (plotdata["POT"]/1e19), fontsize=FONTSIZE, fontweight='bold')

    det = plotdata["det"]
    title = plotdata["title"]
    ax0.set_title(f"$\\bf{{{det}}}$ {title}", fontsize=FONTSIZE+2)
    ld = ax0.legend([line], ["Data\n(ON Beam - OFF)"], frameon=False, loc="upper left", fontsize=10)

    ax0_l0, ax0_hi = ax0.get_ylim()
    ax0.set_ylim([ax0_l0, ax0_hi*2.0])
    
    ax0.legend(fontsize=12, loc="upper right", ncol=2, reverse=True)
    ax0.add_artist(ld)

    chi2_str = "$\\chi^2_\\mathrm{shape}$" if plotdata["areanorm"] else "$\\chi^2$"
    ax0.text(0.05, 0.8, "%s: %.1f / %i" % (chi2_str, plotdata["chi2"], plotdata["ndof"] - int(plotdata["areanorm"])),
            verticalalignment="top", horizontalalignment="left", fontsize=FONTSIZE-2, transform=ax0.transAxes)
    
    plt.subplots_adjust(hspace=0.05)
    return fig, ax0, ax1

In [34]:
for before_cut, step_cn, step_v, step_b, step_l in zip(cutnames[:-1], cutnames[1:], plotvars, bins, labels):
    for v, b, l in zip(step_v, step_b, step_l):
        print(before_cut)
        dat = make_plot_data(v, b, before_cut, "glob_scale", breakdown_mode, True, mode_labels, 
                                  HAWKS_COLORS, l, "Before "+step_cn, fillna=-1, det=DETECTOR)
        ratio_plot(plt, dat)
        savename_pdf = PLOTDIR + "/pdf/Before%s_%s_%s.pdf" % (dat["det"].replace(" ", "-"), step_cn.replace(" ", "").replace(".", "").lower(), v)
        savename_png = PLOTDIR + "/png/Before%s_%s_%s.png" % (dat["det"].replace(" ", "-"), step_cn.replace(" ", "").replace(".", "").lower(), v)
        plt.savefig(savename_pdf, bbox_inches="tight")
        plt.savefig(savename_png, bbox_inches="tight")
        plt.close()
        
        print(step_cn)
        dat = make_plot_data(v, b, step_cn, "glob_scale", breakdown_mode, True, mode_labels, 
                                  HAWKS_COLORS, l, "After "+step_cn, fillna=-1, det=DETECTOR)
        ratio_plot(plt, dat)
        savename_pdf = PLOTDIR + "/pdf/After%s_%s_%s.pdf" % (dat["det"].replace(" ", "-"), step_cn.replace(" ", "").replace(".", "").lower(), v)
        savename_png = PLOTDIR + "/png/After%s_%s_%s.png" % (dat["det"].replace(" ", "-"), step_cn.replace(" ", "").replace(".", "").lower(), v)
        plt.savefig(savename_pdf, bbox_inches="tight")
        plt.savefig(savename_png, bbox_inches="tight")
        plt.close()

No Cuts


/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/../syst.py:170: RuntimeWarning: invalid value encountered in divide
  scale = NCV/NCV_lcl
/tmp/ipykernel_12038/2348737696.py:22: RuntimeWarning: invalid value encountered in divide
  ax1.errorbar(centers, NData/NMC, NDataErr/NMC, color="black", linestyle="none", marker=".")
/tmp/ipykernel_12038/2348737696.py:25: RuntimeWarning: invalid value encountered in divide
  ax1.fill_between(bins[:-1], 1+err/NMC, 1-err/NMC, facecolor="none", hatch="//", edgecolor="gray", linewidth=0.0, step="post")


(<Figure size 640x480 with 2 Axes>,
 <Axes: title={'center': '$\\bf{ICARUS Run4}$ Before FV Cut Step'}, ylabel='Area Normalized'>,
 <Axes: xlabel='Vertex X [cm]'>)

FV Cut Step


/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/../syst.py:170: RuntimeWarning: invalid value encountered in divide
  scale = NCV/NCV_lcl
/tmp/ipykernel_12038/2348737696.py:22: RuntimeWarning: invalid value encountered in divide
  ax1.errorbar(centers, NData/NMC, NDataErr/NMC, color="black", linestyle="none", marker=".")
/tmp/ipykernel_12038/2348737696.py:25: RuntimeWarning: invalid value encountered in divide
  ax1.fill_between(bins[:-1], 1+err/NMC, 1-err/NMC, facecolor="none", hatch="//", edgecolor="gray", linewidth=0.0, step="post")


...

Containment Cut Step


(<Figure size 640x480 with 2 Axes>,
 <Axes: title={'center': '$\\bf{ICARUS Run4}$ Before PID Cut Step'}, ylabel='Area Normalized'>,
 <Axes: xlabel='Muon Cand. Length [cm]'>)

PID Cut Step


/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/../syst.py:170: RuntimeWarning: invalid value encountered in divide
  scale = NCV/NCV_lcl
/tmp/ipykernel_12038/2348737696.py:22: RuntimeWarning: invalid value encountered in divide
  ax1.errorbar(centers, NData/NMC, NDataErr/NMC, color="black", linestyle="none", marker=".")
/tmp/ipykernel_12038/2348737696.py:25: RuntimeWarning: invalid value encountered in divide
  ax1.fill_between(bins[:-1], 1+err/NMC, 1-err/NMC, facecolor="none", hatch="//", edgecolor="gray", linewidth=0.0, step="post")


(<Figure size 640x480 with 2 Axes>,
 <Axes: title={'center': '$\\bf{ICARUS Run4}$ After PID Cut Step'}, ylabel='Area Normalized'>,
 <Axes: xlabel='Muon Cand. Length [cm]'>)

In [ ]:
# def inner(dat):
#     (v, b, l, cut, cutname) = dat
#     return v, make_plot_data(v, b, cutname, "glob_scale", breakdown_mode, True, mode_labels, 
#                                   mode_colors, l, cutname, fillna=-1, det=DETECTOR)

In [36]:

# cuts = [
#     FV,
#     simple_cosmic_rej,
#     twoprong_cut,
#     pid_cut,
# ]

# cutnames = [
#     "Contained",
#     "Simple Cos. Rej.",
#     "Two Prong Cut",
#     "PID Cut",
# ]

# plotvars = [
#     "nu_score",
#     "other_trk_length",
#     "other_shw_length",
#     "mu_chi2_of_prot_cand",
#     "prot_chi2_of_prot_cand",
#     "mu_chi2_of_mu_cand",
#     "prot_chi2_of_mu_cand",  
#     "del_p",
#     "mu_len",

#     "slc_vtx_x",
#     "slc_vtx_y",
#     "slc_vtx_z",
#     "mu_end_x",
#     "mu_end_y",
#     "mu_end_z",
#     "p_end_x",
#     "p_end_y",
#     "p_end_z",
# ]

# bins = [
#     np.linspace(0.25, 0.75, 11),
#     np.array([-5] + list(np.linspace(0, 20, 5))),
#     np.array([-10] + list(np.linspace(0, 100, 11))),
#     np.linspace(0, 60, 13),
#     np.linspace(0, 120, 9),
#     np.linspace(0, 60, 13),
#     np.linspace(0, 300, 21),
#     np.linspace(0, 1, 11),
#     np.linspace(0, 200, 11),
    
#     x_range,
#     y_range,
#     z_range,
#     x_range,
#     y_range,
#     z_range,
#     x_range,
#     y_range,
#     z_range
# ]

# labels = [
#     "$\\nu$ Score",
#     "Maximum Third Track Length [cm]",
#     "Maximum Shower Length [cm]",
#     "Proton Cand. $\\mu$-like PID",
#     "Proton Cand. $p$-like PID",
#     "Muon Cand. $\\mu$-like PID",
#     "Muon Cand. $p$-like PID",
#     "Neutron Momentum [GeV/c]",
#     "Muon Cand. Length [cm]",

#     "Vertex X [cm]",
#     "Vertex Y [cm]",
#     "Vertex Z [cm]",
#     "Muon End X [cm]",
#     "Muon End Y [cm]",
#     "Muon End Z [cm]",
#     "Proton End X [cm]",
#     "Proton End Y [cm]",
#     "Proton End Z [cm]"
# ]

# for c, cname in zip(cuts, cutnames):
#     df[cname] = c(df)
#     df0[cname] = c(df0)
#     df_ooav[cname] = c(df_ooav)
#     ONdf[cname] = c(ONdf)
#     OFFdf[cname] = c(OFFdf)
#     dirt[cname] = c(dirt)
    
#     for i in range(len(detvars)):
#         detvars[i][cname] = c(detvars[i])
        
#     for i in range(len(chi2_detvars)):
#         if isinstance(chi2_detvars[i], list):
#             for j in range(len(chi2_detvars[i])):
#                 chi2_detvars[i][j][cname] = c(chi2_detvars[i][j])
#         else:
#             chi2_detvars[i][cname] = c(chi2_detvars[i])
            
#     for i in range(len(chi2_2x_detvars)):
#         if isinstance(chi2_2x_detvars[i], list):
#             for j in range(len(chi2_2x_detvars[i])):
#                 chi2_2x_detvars[i][j][cname] = c(chi2_2x_detvars[i][j])
#         else:
#             chi2_2x_detvars[i][cname] = c(chi2_2x_detvars[i])

In [37]:
# all_plotdata = {}

# with Pool(20) as p:
#     for cut, cutname in zip(cuts, cutnames):
#         all_plotdata[cutname] = {}
#         inputs = [(v, b, l, cut, cutname) for (v, b, l) in zip(plotvars, bins, labels)]
#         for v, plotdata in tqdm(p.imap_unordered(inner, inputs), total=len(inputs)):    
#             all_plotdata[cutname][v] = plotdata

In [55]:
# This is for TN plots, makes a million files, so shouldn't use by default!
#ifig = 0
#for cname in cutnames:
#    for v in plotvars:
#        plt.figure(ifig)
#        _ = ratio_plot(plt, all_plotdata[cname][v])
#        
#        if DOSAVE:
#            savename_pdf = PLOTDIR + "/pdf/%s_%s_%s.pdf" % (all_plotdata[cname][v]["det"].replace(" ", "-"), cname.replace(" ", "").replace(".", "").lower(), v)
#            savename_png = PLOTDIR + "/png/%s_%s_%s.png" % (all_plotdata[cname][v]["det"].replace(" ", "-"), cname.replace(" ", "").replace(".", "").lower(), v)
#            plt.savefig(savename_pdf, bbox_inches="tight")
#            plt.savefig(savename_png, bbox_inches="tight")
#            plt.close()
#        else:
#            ifig += 1